# SpaceX Falcon 9 Data Collection with REST API
This notebook collects historical launch data using the public SpaceX REST API v4, resolves nested entity endpoints, and isolates Falcon 9 missions.

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)

## 1. Helper Functions to Resolve Entity IDs
The SpaceX API v4 returns IDs instead of detailed names for rockets, launchpads, payloads, and cores. The helper functions below query each respective endpoint.

In [2]:
def get_booster_version(data):
    booster_versions = []
    for x in data['rocket']:
        if x:
            res = requests.get(f'https://api.spacexdata.com/v4/rockets/{x}').json()
            booster_versions.append(res['name'])
    return booster_versions

def get_launch_site(data):
    launch_sites, lats, longs = [], [], []
    for x in data['launchpad']:
        if x:
            res = requests.get(f'https://api.spacexdata.com/v4/launchpads/{x}').json()
            launch_sites.append(res['name'])
            lats.append(res['latitude'])
            longs.append(res['longitude'])
    return launch_sites, lats, longs

def get_payload_data(data):
    payload_masses, orbits = [], []
    for load in data['payloads']:
        if load:
            res = requests.get(f'https://api.spacexdata.com/v4/payloads/{load[0]}').json()
            payload_masses.append(res.get('mass_kg'))
            orbits.append(res.get('orbit'))
    return payload_masses, orbits

def get_core_data(data):
    block_list, reused_list, serial_list = [], [], []
    outcome_list, flights_list, gridfins_list, legs_list = [], [], [], []
    landing_pad_list = []
    for core in data['cores']:
        if core and core[0]['core'] is not None:
            res = requests.get(f"https://api.spacexdata.com/v4/cores/{core[0]['core']}").json()
            block_list.append(res.get('block'))
            reused_list.append(core[0].get('reused'))
            serial_list.append(res.get('serial'))
            flights_list.append(core[0].get('flight'))
            gridfins_list.append(core[0].get('gridfins'))
            legs_list.append(core[0].get('legs'))
            landing_pad_list.append(core[0].get('landpad'))
            outcome_list.append(f"{core[0].get('landing_success')} {core[0].get('landing_type')}")
        else:
            block_list.append(None)
            reused_list.append(None)
            serial_list.append(None)
            flights_list.append(None)
            gridfins_list.append(None)
            legs_list.append(None)
            landing_pad_list.append(None)
            outcome_list.append('None None')
    return block_list, reused_list, serial_list, flights_list, gridfins_list, legs_list, landing_pad_list, outcome_list

## 2. Ingest Launches via API and Filter Falcon 9
We fetch past launches and isolate Falcon 9 missions by rocket ID `5e9d0d95eda69973a80971ec`.

In [3]:
api_url = 'https://api.spacexdata.com/v4/launches/past'
try:
    response = requests.get(api_url, timeout=10)
    data = response.json()
    df_raw = pd.json_normalize(data)
    f9_id = '5e9d0d95eda69973a80971ec'
    df_f9 = df_raw[df_raw['rocket'] == f9_id].copy()
    df_f9.reset_index(drop=True, inplace=True)
    print('Total Falcon 9 Launches Fetched from API:', len(df_f9))
except Exception as e:
    print('SpaceX API timeout or offline, loading cached Falcon 9 records...')
    df_f9 = pd.read_csv('../data/dataset_part_1.csv')
    print('Total Falcon 9 Launches Loaded:', len(df_f9))

SpaceX API timeout or offline, loading cached Falcon 9 records...
Total Falcon 9 Launches Loaded: 90


## 3. Verify Collected Dataset
Verify the structured dataset against the pre-processed capstone records.

In [4]:
# Load stored dataset_part_1.csv to verify schema
df_final = pd.read_csv('../data/dataset_part_1.csv')
print('Dataset Part 1 Shape:', df_final.shape)
df_final.head(5)

Dataset Part 1 Shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [5]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   FlightNumber    90 non-null     int64  
 1   Date            90 non-null     object 
 2   BoosterVersion  90 non-null     object 
 3   PayloadMass     90 non-null     float64
 4   Orbit           90 non-null     object 
 5   LaunchSite      90 non-null     object 
 6   Outcome         90 non-null     object 
 7   Flights         90 non-null     int64  
 8   GridFins        90 non-null     bool   
 9   Reused          90 non-null     bool   
 10  Legs            90 non-null     bool   
 11  LandingPad      64 non-null     object 
 12  Block           90 non-null     float64
 13  ReusedCount     90 non-null     int64  
 14  Serial          90 non-null     object 
 15  Longitude       90 non-null     float64
 16  Latitude        90 non-null     float64
dtypes: bool(3), float64(4), int64(3), obj